# Prediction Overview
**APPROACH · RESULTS · V1 → V2 · KEY INSIGHT**

---


## Inhalt

- [What We Predict](#what-we-predict)
- [The Prediction Scenario](#the-prediction-scenario)
  - [Was der User eingibt](#was-der-user-eingibt)
  - [Was das Modell daraus macht](#was-das-modell-daraus-macht)
  - [Zur Fahrtrichtung](#zur-fahrtrichtung)
- [Why Machine Learning?](#why-machine-learning)
- [Data Basis](#data-basis)
  - [Note — Analysis vs. Model](#note-analysis-vs-model)
- [The Model — LightGBM](#the-model-lightgbm)
- [Model Comparison — Why LightGBM?](#model-comparison-why-lightgbm)
  - [One-Hot Encoding — Why We Don't Need It](#one-hot-encoding-why-we-dont-need-it)
  - [Early Stopping — How the Model Stops Itself](#early-stopping-how-the-model-stops-itself)
  - [Hyperparameter Tuning and Optuna — Why We Deliberately Skipped It](#hyperparameter-tuning-and-optuna-why-we-deliberately-skipped-it)
- [Metrics](#metrics)
  - [Warum MAE als Hauptmetrik?](#warum-mae-als-hauptmetrik)
- [Modelling Approach](#modelling-approach)
- [Notebooks in This Phase](#notebooks-in-this-phase)
- [Key Decisions](#key-decisions)
- [Baseline — Starting Point](#baseline-starting-point)
- [Success Criteria — Results](#success-criteria-results)
- [Results — LightGBM v1](#results-lightgbm-v1)
  - [Modell-Parameter](#modell-parameter)
  - [Metrics](#metrics)
  - [Fehleranalyse (Test-Set 2025)](#fehleranalyse-test-set-2025)
  - [Known Weaknesses](#known-weaknesses)
  - [Live-Szenario](#live-szenario)
- [v1 → v2: Diagnosis and Iterative Improvement](#v1-v2-diagnosis-and-iterative-improvement)
- [The Signal, Not the Algorithm — Key Insight](#the-signal-not-the-algorithm-key-insight)


Three models built, one key insight: **the signal matters more than the algorithm.**
LightGBM v1 reduced MAE from 50.0s (baseline) to 45.7s. Adding two features in v2 brought it to **18,56 s — a 63% improvement** without changing the algorithm or tuning hyperparameters.


## What We Predict


**Zielvariable: `arrival_delay` in Sekunden** — wie verspätet ist eine Tram wenn sie an einer Haltestelle ankommt?

- **Aufgabe:** Regression (kontinuierlicher Wert, kein Ja/Nein)
- **Metrik:** MAE (Mean Absolute Error) — durchschnittlicher Fehler in Sekunden
- **Ziel:** MAE deutlich unter dem Basis-Netzschnitt von ~56s

**Warum Regression, nicht Klassifikation (OTP Ja/Nein)?**
- Regression gibt mehr Information: "32s Verspätung" ist nützlicher als "pünktlich"
- OTP-Klasse kann jederzeit aus der Regression abgeleitet werden: `arrival_delay > 120s → verspätet`
- Für das spätere Dashboard ist der konkrete Sekundenwert das bessere Input

## The Prediction Scenario


**Frage, die das Modell beantwortet:**
> *„Ich will am Donnerstag um 21:30 Uhr mit der Linie 11 zur Haltestelle Balgrist — wie viel Verspätung muss ich einkalkulieren?"*

---

### Was der User eingibt

| Input | Beispiel | Woher |
|:---|:---|:---|
| **Datum** | Donnerstag, 5. Juni 2026 | Kalender — 14 Tage in die Zukunft |
| **Uhrzeit** | 21:30 Uhr | Nutzer |
| **Haltestelle** | Balgrist | Nutzer wählt aus Liste |
| **Linie** | Linie 11 | Nutzer wählt aus Liste |
| **Wetterlage** | Regen, 12°C | Wettervorhersage API (z.B. Open-Meteo) |
| **Event** | Keine / Klein / Mittel / Gross | Nutzer wählt Cluster — oder automatisch aus Eventkalender |

### Was das Modell daraus macht

Das Modell übersetzt die Nutzer-Eingaben in die 40 Feature-Spalten die es beim Training gesehen hat:

```
hour=21, weekday=3 (Do), month=6, season=2 (Sommer), is_weekend=False
line_name="11", stop_name="Balgrist", district_nr=8
has_rain=True, temperature=12, precipitation=1.4
has_event=False, event_weight=0
...
```

→ Ausgabe: **Erwarteter `arrival_delay` in Sekunden** + OTP-Klasse (pünktlich / verspätet)

---

### Zur Fahrtrichtung

Die Fahrtrichtung ist im MVP **implizit** — das Modell kennt Haltestelle + Linie und hat gelernt, dass Balgrist auf Linie 11 ein Endkorridors-Stop ist mit hohem Delay. Explizite Richtungseingabe (→ Zürich HB vs. → Auzelg) ist eine v2-Erweiterung: dafür müsste `terminus` (letzter Stop des Trips) als Feature ergänzt werden. Der Effekt auf die MAE ist laut `03_analysis_4-spatial.ipynb` messbar aber nicht dramatisch — gut für Iteration 2.

## Why Machine Learning?


Die EDA hat gezeigt: einfache Formeln reichen nicht.

- **Wetter-Korrelationen** sind nicht-linear: max. r = 0.03 (Pearson) — aber Schnee kostet +54s. Der Effekt existiert, ist aber nicht proportional
- **Interaktionen** erklären mehr als Einzelfaktoren: `hour × event_weight` ist stärker als beides allein (F-EVNT-03)
- **Ortsspezifische Effekte**: Linie 11 verhält sich fundamental anders als Linie 6 — nicht durch eine Formel darstellbar
- **Schwellenwerteffekte**: Schnee ab einer gewissen Intensität, Events ab 18h — klassische Baummodell-Stärke

→ Ein Modell das Entscheidungsbäume baut (LightGBM) kann all das lernen.

## Data Basis


| Datei | Zeilen | Verwendung |
|:---|---:|:---|
| `train_final.parquet` | 55.5 Mio. | Modell trainieren |
| `test_final.parquet` | ~25 Mio. | Modell evaluieren — unberührt bis zum Schluss |

**Split-Strategie:** Temporal — 2023–2024 Train / 2025 Test. Kein Random-Shuffle, weil das Datenleck erzeugen würde (Daten vom gleichen Tag würden in Train und Test landen).

**32 Features im Modell** — definitive Liste in `data/models/lgbm_v1_meta.json`:

| Gruppe | Anzahl | Features |
|:---|:---:|:---|
| **Netz** | 7 | `line_name` · `stop_name` · `district_nr` · `n_lines_at_stop` · `n_stops_line` · `is_start_stop` · `is_end_stop` |
| **Zeit** | 8 | `hour` · `weekday` · `month` · `year` · `season` · `is_weekend` · `is_november` · `is_late_night_weekend` |
| **Wetter** | 9 | `temperature` · `precipitation` · `wind_speed` · `flood_intensity` · `has_rain` · `has_heavy_rain` · `has_snow` · `has_flood` · `is_hot` |
| **Events** | 6 | `event_type` · `event_size` · `is_holiday` · `has_event` · `event_weight` · `event_weight_x_hour` |
| **Fahrplan** | 2 | `dwell_time` · `gtfs_year` |

**5 kategoriale Spalten** (LightGBM Native-Categorical): `line_name` · `stop_name` · `event_type` · `season` · `gtfs_year`

**Bewusst ausgeschlossen:**

| Spalte | Grund |
|:---|:---|
| `departure_delay` · `delay_delta` | Leakage — kennen wir erst wenn die Tram schon da ist |
| `operating_date` · `trip_id` | Identifier, kein inhaltliches Feature |
| `stop_lat` · `stop_lon` | Durch `stop_name` + `district_nr` abgedeckt |
| `event_name` · `event_location` | Zu granular, zu viele unique Values |
| `canceled` | Target-nahe Information, kein Prediction-Feature |

### Note — Analysis vs. Model

The **analysis notebooks** (`03_analysis_*`) use `lf_all = concat(TRAIN, TEST)` — all 3 years, full dataset via Polars LazyFrame. This is methodologically correct: EDA describes reality, no model sees the future during exploration.

The **model** only sees `train_final.parquet` (2023–Jun 2024). The test split (2025) remains untouched until final evaluation.


## The Model — LightGBM


**LightGBM** ist ein Gradient-Boosting-Algorithmus — er baut viele kleine Entscheidungsbäume, die nacheinander die Fehler des vorherigen Baums korrigieren.

**Warum LightGBM, nicht XGBoost oder Random Forest?**
- Deutlich schneller auf großen Datensätzen (55 Mio. Zeilen)
- Kann kategoriale Features (`line_name`, `stop_name`, `event_type`) direkt verarbeiten — kein manuelles Encoding nötig
- Feature Importance direkt eingebaut → erklärt was das Modell gelernt hat
- Sehr gute Out-of-the-box Performance, wenig Tuning zum Start nötig

**Wie lernt das Modell?**
1. Es nimmt alle 55 Mio. Trainingszeilen
2. Baut einen ersten einfachen Baum: "wenn hour > 20 und has_snow → höherer Delay"
3. Schaut was noch falsch ist (Residuen)
4. Baut nächsten Baum der die Fehler des ersten korrigiert
5. Wiederholt das 200–1000 Mal
6. Finale Vorhersage = Summe aller Bäume

**Encoding-Entscheidungen (vor dem Training zu treffen):**
- `stop_name` (6.000+ unique) → LightGBM Native-Categorical oder Target-Encoding mit n-Threshold ≥ 1.000
- `line_name`, `event_type`, `gtfs_year` → LightGBM Native-Categorical
- `season`, `weekday`, `month` → bereits numerisch, kein Encoding nötig

## Model Comparison — Why LightGBM?


| Modell | Nicht-linear | Schnell (55 Mio.) | Native Categoricals | Feature Importance | Unser Urteil |
|:---|:---:|:---:|:---:|:---:|:---|
| **Lineare Regression** | ❌ | ✅ | ❌ | ✅ | Zu schwach — Korrelationen max. r=0.03 |
| **Ridge / Lasso** | ❌ | ✅ | ❌ | ✅ | Nur als Baseline-Kandidat |
| **Random Forest** | ✅ | ❌ | ❌ | ✅ | Zu langsam auf 55 Mio. Zeilen |
| **XGBoost** | ✅ | 🟡 | ❌ | ✅ | Gut, aber langsamer als LightGBM |
| **LightGBM** | ✅ | ✅ | ✅ | ✅ | **Primärmodell** |
| **CatBoost** | ✅ | 🟡 | ✅✅ | ✅ | Alternative wenn stop_name problematisch |

**Die drei entscheidenden Kriterien für unser Projekt:**

1. **Nicht-linear** — EDA zeigt Schwellenwerteffekte (Schnee, Events ab 18h) → lineare Modelle können das nicht lernen
2. **Schnell auf 55 Mio. Zeilen** — LightGBM arbeitet mit Histogramm-Splitting statt alle Datenpunkte einzeln zu prüfen → 5–10× schneller als XGBoost auf großen Daten
3. **Native Categoricals** — `stop_name` hat 6.000+ unique Values, `line_name` 16. LightGBM kann Kategoricals direkt verarbeiten — kein aufwändiges Encoding nötig, das Overfitting riskiert

**Wie Gradient Boosting funktioniert (vereinfacht):**
- Baum 1 sagt: „Ø 56s für alle" → Fehler: mal zu viel, mal zu wenig
- Baum 2 lernt die Fehler von Baum 1: „Bei Schnee und L9 ist Baum 1 60s zu niedrig" → korrigiert
- Baum 3 lernt die Fehler von Baum 1+2 → korrigiert weiter
- Nach 300–1000 Bäumen: Summe aller Korrekturen = präzise Vorhersage

### One-Hot Encoding — Why We Don't Need It

**The problem with text categories in classical models:**

Algorithms work with numbers. A column `stop_name = "Paradeplatz"` cannot be read directly by a tree model. The standard solution is **One-Hot Encoding (OHE)**:

> For each category, a separate 0/1 column is created.
> "Paradeplatz" → column `stop_Paradeplatz = 1`, all others = 0

With 500+ stops, OHE creates 500+ columns — sparse, slow, memory-intensive.

**LightGBM handles categoricals natively** — no encoding needed. It splits on category values directly, keeps one column, and is faster and more accurate as a result.


### Early Stopping — How the Model Stops Itself

LightGBM builds trees iteratively: round 1, round 2, ... up to `n_estimators=1000`. But more rounds do not automatically mean better results.

**Without stopping:** beyond a certain point the model no longer learns new real patterns — it starts memorising the training data (**overfitting**). Training performance improves, validation performance degrades.

**Early Stopping:** after each round, validation MAE is checked. If it has not improved after 50 consecutive rounds, training stops. This is where v1 stopped at **iteration 481**.


### Hyperparameter Tuning and Optuna — Why We Deliberately Skipped It

**What are hyperparameters?**

Hyperparameters are the model's configuration settings that must be fixed before training — the model does not learn them from data:
- `num_leaves=63` — how complex may the trees be?
- `learning_rate=0.05` — how strongly does each new tree correct?
- `feature_fraction=0.8` — what fraction of features is sampled per tree?

**Why we skipped Optuna:** v1 already showed that the bottleneck was not the hyperparameters — it was a missing feature (`prev_trip_delay`). Optimising knobs on a model with a structural gap wastes compute. The hypothesis was correct: v2 with the same hyperparameters and two new features cut MAE from 45.7s to 18,56 s.


## Metrics


| Metrik | Einheit | Formel (vereinfacht) | Stärke | Schwäche | Wir nutzen |
|:---|:---|:---|:---|:---|:---:|
| **MAE** | Sekunden | Ø \|Fehler\| | Intuitiv: „Im Schnitt X Sekunden daneben" | Behandelt alle Fehler gleich | ✅ Primär |
| **RMSE** | Sekunden | √(Ø Fehler²) | Bestraft große Ausreisser stärker | Schwerer zu interpretieren | ✅ Sekundär |
| **R²** | 0–1 | 1 − (Fehler / Varianz) | Zeigt: wie viel erklärt das Modell? | Bei schiefer Verteilung irreführend | ℹ️ Orientierung |
| **OTP-Accuracy** | % | Anteil \|Fehler\| ≤ 60s | Business-relevant | Ignoriert Größe des Fehlers | ✅ Ergänzend |
| **MBE** | Sekunden | Ø (Pred − Ist) | Zeigt systematischen Bias | Pos./neg. heben sich auf — kein Genauigkeitsmaß | ✅ Ergänzend |
| **MAPE** | % | Ø \|Fehler / Ist-Wert\| | Prozentual verständlich | Explodiert bei delay ≈ 0s | ❌ |

### Warum MAE als Hauptmetrik?

MAE ist in **Sekunden** — ein Fehler von 30s ist auch 30s Abweichung im Report. Das ist direkt kommunizierbar, auch für Nicht-Techniker.

**Warum nicht RMSE?** RMSE bestraft große Fehler quadratisch. Bei Transit-Delays gibt es strukturelle Ausreißer — Unfälle, Totalausfälle, extreme Wetterlagen. Diese seltenen Extremfälle sind operativ nicht durch Fahrplandesign beeinflussbar. RMSE würde das Modell auf diese Extremfälle optimieren und dabei die alltäglichen Delays vernachlässigen, die tatsächlich steuerbar sind.

**Was ist MBE?** Mean Bias Error = durchschnittlicher Fehler *mit Vorzeichen*, nicht absolut. MBE +8,3 s bedeutet: das Modell sagt im Schnitt 8.3 Sekunden zu wenig — es unterschätzt systematisch. Ein MBE nahe 0 ist Zeichen guter Kalibrierung. MBE ergänzt MAE: MAE misst Präzision, MBE misst Richtung des Fehlers.

**Warum MAPE nicht?** Bei frühen Abfahrten (delay = −30s) oder pünktlichem Tram (delay = 0–5s) geht der prozentuale Fehler gegen unendlich. Das macht MAPE auf diesem Datensatz unbrauchbar.

## Modelling Approach


**Step 1 — Baseline** (`06_prediction_1-baseline.ipynb`)
Four baseline candidates evaluated (grand mean → stop mean). Best baseline: **Stop Mean at MAE 50.0s**. This is the benchmark every model must beat.

**Step 2 — LightGBM v1** (`06_prediction_2-model.ipynb`)
32 features, LightGBM native categoricals, temporal validation split (Jul–Dec 2024). Training on 41.2M rows. Best iteration: 481 (early stopping). **Validation MAE: 46.4s · Test MAE: 45.7s.**

**Step 3 — Evaluation + Error Analysis** (`06_prediction_3-evaluation.ipynb`)
Feature importance, residual analysis, error by segment. Key finding: MBE +8,3 s — model systematically under-predicts. Root cause: cascade effect not captured — `prev_trip_delay` missing.

**Step 4 — LightGBM v2** (`06_prediction_4-model_v2.ipynb`)
Two new features: `prev_trip_delay` (cascade) + `stop_sequence_pct` (position in route). Same hyperparameters. **Test MAE: 18,56 s · MBE: −0,69 s** — bias resolved, 63% improvement.

**Step 5 — XGBoost Robustness Check** (`06_prediction_5-comparison.ipynb`)
XGBoost as cross-validation of results. Confirms: LightGBM is superior on this dataset (speed + accuracy).


## Notebooks in This Phase


| Notebook | Content |
|:---|:---|
| `06_prediction_0-overview` | This overview — approach, decisions, results summary |
| `06_prediction_1-baseline` | Feature set · 4 baseline candidates · benchmark: Stop Mean MAE 50.0s |
| `06_prediction_2-model` | LightGBM v1 training · validation · MAE 45.7s |
| `06_prediction_3-evaluation` | Test evaluation · feature importance · error analysis · MBE diagnosis |
| `06_prediction_4-model_v2` | LightGBM v2 · 2 new features · MAE 18,56 s · SHAP analysis |
| `06_prediction_5-comparison` | XGBoost robustness check · full model comparison |
| `06_prediction_6-dwell_simulator` | Dwell time confounding analysis · F-SIM-01–04 |


## Key Decisions


| Decision | Choice | Rationale |
|:---|:---|:---|
| `stop_name` encoding | **LightGBM Native-Categorical** | No manual encoding needed; faster, no 500+ OHE columns |
| `departure_delay` as feature | **Not used** | Leakage risk — would require knowing departure delay at prediction time |
| Cascade feature | **`prev_trip_delay`** | Strongest single feature in v2 (see F-NET-07); drove the 45.7s → 18,56 s jump |
| Sampling | **All 55M rows** | No subsampling — LightGBM handles the full dataset efficiently |
| Validation strategy | **Temporal split** | Jul–Dec 2024 as validation (same year-gap logic as train/test split) |
| Hyperparameter tuning | **Skipped (Optuna)** | Bottleneck was a missing feature, not hyperparameters — confirmed by v2 result |


## Baseline — Starting Point


**Eine Baseline ist der erste rohe Wurf — die dümmste sinnvolle Vorhersage.** Kein Training, kein Modell, nur Durchschnittswerte aus den Trainingsdaten. Jedes echte Modell das diese Zahl nicht schlägt ist nutzlos.

**Vier Baseline-Kandidaten (von simpel zu stark):**

| # | Baseline | Logik | Erwarteter MAE |
|:---|:---|:---|:---|
| 1 | **Grand Mean** | Immer 56s vorhersagen (Netzschnitt) | ~45s |
| 2 | **Stunden-Mittelwert** | 7h → 49s, 21h → 68s — je nach Uhrzeit | ~38s (Schätzung) |
| 3 | **Linien-Mittelwert** | L11 → 69s, L6 → 38s — je nach Linie | ~32s (Schätzung) |
| 4 | **Stop-Mittelwert** | Historischer Ø pro Haltestelle | ~25s (Schätzung) |

→ Der **Stop-Mittelwert** ist der härteste Gegner. Das LightGBM-Modell muss ihn schlagen — sonst hat es nichts gelernt was der Stop-Durchschnitt nicht schon weiß.

**Warum zuerst Baseline, dann Modell?**
- Ohne Baseline weiß man nicht ob MAE=32s gut oder schlecht ist
- Baseline schützt vor False Confidence: ein Modell das nur den Stunden-Durchschnitt lernt wirkt beeindruckend — ist aber keine echte Intelligenz
- Portfolio: zeigen dass man sauber vorgeht ist professioneller als blind ein Modell zu trainieren

## Success Criteria — Results


| Threshold | Interpretation | Actual |
|:---|:---|:---|
| MAE > 45s | Worse than naive baseline — something is wrong | v1: 45.7s ✅ barely better |
| MAE 30–45s | Better than baseline, but room to improve | — |
| MAE 20–30s | Good result — model learns real patterns | — |
| MAE < 20s | Very good — or check for overfitting | **v2: 18,56 s ✅ no overfitting** |


## Results — LightGBM v1


### Modell-Parameter

| Parameter | Wert |
|:---|:---|
| Modell | LightGBM v1 |
| Gespeichert | `data/models/lgbm_v1.txt` · `lgbm_v1_meta.json` |
| Features | 32 (siehe Datenbasis-Sektion) |
| Kategoriale Features | 5 (LightGBM Native-Categorical) |
| Train-Zeilen | 41.2 Mio. (2023–Jun 2024) |
| Validation-Zeilen | 14.3 Mio. (Jul–Dez 2024) |
| Beste Iteration | 481 (Early Stopping nach 50 Runden ohne Verbesserung) |
| `num_leaves` | 63 |
| `learning_rate` | 0.05 |
| `feature_fraction` | 0.8 |
| `bagging_fraction` | 0.8 · `bagging_freq` 5 |
| `min_child_samples` | 50 |

### Metrics

| Metrik | Baseline (Stop Mean) | LightGBM v1 | Gewinn |
|:---|---:|---:|---:|
| **MAE Test** | 50.0s | **46.3s** | **−3.7s** |
| **MAE Val** | — | 49.05s | — |
| RMSE Test | 86.2s | ~85s | — |
| OTP ±60s | 73.1% | 75.4% | +2.3 pp |
| MBE Test | — | +8,3 s–+10.1s | — (systematisch zu optimistisch) |

### Fehleranalyse (Test-Set 2025)

**Nach Stunde** — schlechteste Stunden:

| Stunde | MAE |
|:---|:---|
| 17h | 54.4s |
| 16h | 53.9s |
| 18h | 52.4s |

**Nach Linie:**

| Linie | MAE | Linie | MAE |
|:---|:---|:---|:---|
| L11 | 52.5s (schlechteste) | L12 | 34.5s (beste) |
| L8 | 52.2s | L6 | 37.3s |
| L15 | 51.0s | L17 | 40.1s |

**Nach Wetter:**

| Bedingung | MAE |
|:---|:---|
| Schnee | 58.9s (n = 39.920) |
| Regen | 50.3s |
| Normal | 45.9s |

### Known Weaknesses

| Schwäche | Beschreibung | Verbesserung v2 |
|:---|:---|:---|
| MBE +8–10s | Modell unterschätzt systematisch — Extremverspätungen werden zu niedrig vorhergesagt | Target Encoding für `stop_name` statt Native-Cat |
| `stop_name` als Native-Cat | ~500 Stops grob gebündelt — Stop-spezifische Muster unvollständig gelernt | Target Encoding mit n-Threshold ≥ 1.000 |
| `prev_trip_delay` fehlt | Kaskadeneffekt nicht modelliert — ob die Vorgängerfahrt Verspätung hatte, ist ein starkes Signal | Trip-Kontinuität in Datensatz prüfen, dann als Feature |
| Schnee-Schwäche | MAE bei Schnee 13s höher als normal — seltene Extremlagen unterrepräsentiert im Training | Oversampling Schnee-Tage oder separate Schnee-Komponente |

### Live-Szenario

> Dienstag · 17:00 Uhr · Haltestelle Paradeplatz · Linie 11 · leichter Regen

**→ Vorhergesagter Delay: 48s**

Zum Vergleich: Netzschnitt 55s — Modell sagt Paradeplatz ist besser als Netzschnitt, was mit den EDA-Findings übereinstimmt (Paradeplatz 48.2s historisch).

## v1 → v2: Diagnosis and Iterative Improvement

The jump from v1 to v2 is not luck — it follows a clear process:

**v1 — first run (proof of concept)**
- MAE 45.7s → beats baseline (50.0s) ✅
- MBE +8,3 s → model is **systematically too optimistic**: it predicts 8,3 s too low on average
- Weakness identified: `prev_trip_delay` missing — cascade effect not modelled

**What is MBE and why does it matter?**
MAE tells us *how large* the errors are. MBE (Mean Bias Error) tells us *which direction* they go. A positive MBE (+8,3 s) means the model consistently under-predicts delay — it is too optimistic. This is a structural gap, not noise.

**v2 — targeted feature engineering**
- Added `prev_trip_delay`: was the previous tram on this route already late? (cascade signal)
- Added `stop_sequence_pct`: where is this stop in the route (0–1)? (position signal)
- Same hyperparameters, same model architecture
- Result: MAE 18,56 s · MBE −0,69 s — bias resolved, 63% improvement


## The Signal, Not the Algorithm — Key Insight

A common misconception in ML projects:

> *"Better results come from a better algorithm or better tuning."*

This project proves the opposite:

| What changed | MAE |
|:---|---:|
| Algorithm: LightGBM (v1) | 45.7s |
| Algorithm: **same**, hyperparameters: **same**, only 2 new features | **18,56 s** |

The 63% improvement came entirely from understanding the domain: tram delays cascade. A tram that is already late will arrive late at the next stop. This is not a statistical trick — it is operational reality, captured as `prev_trip_delay`.

**Analysis dictates the model.** F-NET-07 (cascade effect, identified in network analysis) became the strongest feature in v2. The structured finding system paid off.
